# 19 - Generate LLM Context from Well Logs

**Part B - Integrated & Advanced Workflows**

This lesson turns LAS data into compact, auditable JSON context packages organized by analysis category, well, formation, and lithology group. It uses only public `petrophys` APIs and bundled sample data, so it runs without machine-specific paths.

The generated JSON is model-neutral: use it with any LLM client, retrieval system, or review workflow.

## Learning goals

- Build or save single-well context after running an interpretation.
- Generate computed category/formation/lithology packages.
- Read ranked CSV or NPD/SODIR Excel tables directly.
- Enforce byte budgets and inspect provenance, warnings and limitations.
- Build a model-neutral request without making a network call.

## What the exporter produces

For each lithology group, the exporter writes separate views for hydrocarbon, saturation, permeability, porosity, numerical statistics, and lithology analysis. Each JSON file records the source well, formation interval, input logs, computed logs, statistics, analysis results, data completeness, and bad-hole QC.

Raw depth-by-depth traces are intentionally excluded by default. This keeps context smaller while retaining the statistics and provenance needed for interpretation and review.

In [ ]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory
import json

import pandas as pd
import petrophys as pp

print("petrophys version:", pp.__version__)
well = pp.read(pp.sample("30_6-5"))
result = pp.interpret(well, rw=0.070, rw_meas_temp_f=230.0).raise_for_errors()
context = pp.build_context(result)
assert context["schema_version"] == "1.0"
assert context["interpretation"] is not None
with TemporaryDirectory() as directory:
    saved = pp.export_context(result, Path(directory) / "context.json")
    print("Computed well context:", saved.name, saved.stat().st_size, "UTF-8 bytes")

## 1. Configure package paths once

Only the LAS is mandatory. This package example also supplies the standard well-parameter, formation, and lithology inputs. All generated files go below a fresh temporary run directory; no private workbook or earlier context package is read.

In [ ]:
WELL_NAME = "30_6-5"
run_directory = TemporaryDirectory(prefix="petrophys-context-demo-")
RUN_ROOT = Path(run_directory.name)
OUTPUT_DIR = RUN_ROOT / "context"  # Must not exist before export.
LAS_FILE = Path(pp.sample(WELL_NAME))
FORMATION_SOURCE = Path(pp.sample("formations_30_6-5"))
LITHOLOGY_SOURCE = Path(pp.sample("lithology_30_6-5"))
PARAMETERS_SOURCE = RUN_ROOT / "well_parameters.csv"
INPUT_DIR = LAS_FILE.parent
MINIMUM_GROUP_THICKNESS = 2.0
MAX_BYTES = 1_000_000

# This row follows well_parameters_template.csv. Values are illustrative,
# not a validated reservoir calibration.
pd.DataFrame([{
    "well": WELL_NAME, "lithology_domain": "auto",
    "rw": 0.070, "rw_unit": "ohm.m",
    "rw_reference_temperature": 230.0, "rw_temperature_unit": "F",
    "gr_clean": 20.0, "gr_shale": 100.0,
}]).to_csv(PARAMETERS_SOURCE, index=False)
print("Lithology:", LITHOLOGY_SOURCE.name)
print("Parameters:", PARAMETERS_SOURCE.name)
print("Domain:", pp.read_well_parameters(PARAMETERS_SOURCE)["lithology_domain"])

print("LAS:", LAS_FILE.name)
print("Formations:", FORMATION_SOURCE.name)


For your own data, fill the packaged templates and replace these paths. CSV/TSV/Excel tables are accepted, including NPD/SODIR `Well identifier` / `Surface` / `MD` headers. Formations, lithology, core, and codebooks are optional. Supply sheet and depth metadata when needed.

## 2. Inspect the ranked formation table

The sample declares metre depths, explicit bottoms, rank and parent. Groups intentionally overlap their formations. Select `rank="formation"` for this package; do not infer hierarchy by deleting geometrically nested rows. No temporary tops workbook or guessed final bottom is needed.

In [ ]:
formations = pp.read_formations(FORMATION_SOURCE)
display(formations[["formation_name", "top_depth", "bottom_depth", "rank", "parent"]])
assert set(formations["depth_unit"]) == {"m"}
assert {"group", "formation"}.issubset(set(formations["rank"]))
print("Selected formation intervals:", int((formations["rank"] == "formation").sum()))

## 3. Generate bounded, category-first context

Package exports read the standardized inputs, run the workflow, group intervals, and validate all files before publishing the new directory. Category documents contain the computed context directly, without replay or a wrapper object. A failed or oversized export raises an exception without leaving a partial destination.

In [ ]:
results = pp.export_context(
    INPUT_DIR, OUTPUT_DIR,
    layout="category_formation_lithology",
    parameters_file=PARAMETERS_SOURCE,
    tops_file=FORMATION_SOURCE,
    lithology_file=LITHOLOGY_SOURCE,
    tops_options={"rank": "formation"},
    pattern=LAS_FILE.name,
    recursive=False,
    minimum_group_thickness=MINIMUM_GROUP_THICKNESS,
    well_names=[WELL_NAME],
    max_bytes=MAX_BYTES,
    max_total_bytes=50_000_000,
)
print("Generated records:", len(results))
assert results and all(item.success for item in results)

## 4. Summarize and review warnings

The API has checked failures. Still inspect warnings before interpreting the exported statistics; successful serialization is not scientific validation.

In [ ]:
context_files = sorted({Path(item.json_file) for item in results})
category_counts = Counter(path.relative_to(OUTPUT_DIR).parts[0] for path in context_files)
summary = pd.DataFrame({
    "category": list(category_counts),
    "json_files": list(category_counts.values()),
}).sort_values("category", ignore_index=True)
display(summary)
warnings = sorted({warning for item in results for warning in item.warnings})
print("Export warnings:", len(warnings))
for warning in warnings[:5]:
    print(warning)
assert all(path.stat().st_size <= MAX_BYTES for path in context_files)
print("JSON context files:", len(context_files))

In [ ]:
# Show a compact, relative file listing instead of machine-specific paths.
for path in context_files[:12]:
    print(path.relative_to(RUN_ROOT))
if len(context_files) > 12:
    print(f"... and {len(context_files) - 12} more JSON files")

## 5. Inspect one context document

Choose the category that matches the question being asked. Here we open a hydrocarbon context and inspect only high-signal metadata from the complete document.

In [ ]:
hydrocarbon_files = [
    path for path in context_files
    if path.relative_to(OUTPUT_DIR).parts[0] == "hydrocarbon"
]
context_path = hydrocarbon_files[0] if hydrocarbon_files else context_files[0]
document = json.loads(context_path.read_text(encoding="utf-8"))
analysis = document["analysis_results"]
overview = pd.Series({
    "file": str(context_path.relative_to(RUN_ROOT)),
    "well": analysis["well_name"],
    "formation": analysis["formation_name"],
    "lithology_group": analysis["lithology group"],
    "lithology": analysis["lithology"],
    "input_log_count": len(analysis["input_logs_present"]),
    "derived_log_count": len(analysis["derived_logs_computed"]),
    "raw_traces_included": document["include_raw_traces_in_context"],
})
display(overview.to_frame("value"))

In [ ]:
# Basic schema and provenance checks before an LLM sees the document.
required_top_level = {
    "formation_info",
    "analysis_results",
    "raw_log_traces_in_segment",
    "include_raw_traces_in_context",
}
required_analysis = {
    "well_name",
    "formation_name",
    "statistics",
    "analysis_summary",
    "data_completeness",
    "input_logs_present",
    "derived_logs_computed",
    "BADHOLE_QC_CHECK",
}

assert required_top_level.issubset(document)
assert required_analysis.issubset(analysis)
assert analysis["well_name"]
assert document["formation_info"]["thickness"] >= 0
print("Schema and provenance checks passed.")

### Scientific review, beyond schema

Inspect a known interval and the assumptions used to calculate it. The following numerical checks only verify valid ranges and the explicit Rw basis. They do not establish hydrocarbons, validate the chosen calibration, or justify shale/overburden pay flags. Review clay-volume methods, bad-hole QC, saturation-model assumptions, and supplied geology before treating cutoff screening as pay.

In [ ]:
etive_sand = [
    path for path in context_files
    if path.relative_to(OUTPUT_DIR).parts[0] == "saturation"
    and any("etive" in part.lower() for part in path.parts)
    and "SAND" in path.name
]
assert etive_sand, "No Etive sand context found; review input selection"
probe = json.loads(etive_sand[0].read_text(encoding="utf-8"))
stats = probe["analysis_results"]["statistics"]
sw_mean = stats["SW_overall"]["arithmetic_mean"]
rw_mean = stats["RW_overall"]["arithmetic_mean"]
print(f"{etive_sand[0].name}")
print(f"SW mean: {sw_mean:.3f}; corrected Rw mean: {rw_mean:.3f} ohm.m")
assert 0 <= sw_mean <= 1
assert rw_mean > 0
print("Numerical checks passed; reservoir calibration still requires review.")

## 6. Build a model-neutral LLM request

Keep measured inputs, computed outputs, warnings, and missing-data statements intact. Ask the model to distinguish observation from inference and to avoid inventing values. The object below can be serialized for any LLM SDK or stored in a retrieval index.

In [ ]:
llm_request = {
    "task": "Summarize the interval's measured evidence, calculations and limitations.",
    "instructions": [
        "Use only the supplied context.",
        "Separate measurements from calculated interpretations.",
        "Do not infer missing values or equate cutoff screening with confirmed pay.",
        "Report well, formation, lithology group and depth interval.",
        "Retain QC warnings and state which assumptions need calibration.",
    ],
    "petrophysical_context": document,
}
serialized_request = json.dumps(llm_request, ensure_ascii=False)
print("Request UTF-8 bytes:", len(serialized_request.encode("utf-8")))
print("Prepared locally; no LLM or network call was made.")
run_directory.cleanup()

## CLI equivalent

```bash
petrophys-context well.las context.json --parameters well_parameters.csv --formations formation_intervals.csv --lithology lithology_intervals.csv --core core_measurements.csv
petrophys-context logs outputs/context_run_01 --layout category_formation_lithology --parameters well_parameters.csv --formations formation_intervals.csv --lithology lithology_intervals.csv --core core_measurements.csv --tops-rank formation
```

Remove flags for optional inputs that are unavailable. The command always computes context from the LAS and supplied standardized tables. It never reads or copies an earlier context package.

## Recap

- `pp.build_context(result)` and `pp.export_context(result, path)` work without geology.
- The CLI always interprets LAS input before exporting context.
- Package exports accept standardized parameters plus optional geology and core tables.
- Exports are byte-bounded and package directories are never overwritten.
- Warnings and physical calibration matter beyond JSON/schema checks.
- Context generation is model-neutral and makes no LLM request.